macro_monthly_oecd_final.py

In [1]:
import pandas as pd
import pandas_datareader.data as web
import requests
import io
import datetime
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

start = datetime.datetime(1996, 1, 1)
end = datetime.datetime.today()

# ============================================
# 0. USDKRW (FRED โดยตรง — ไม่มีปัญหา ไม่ต้องเปลี่ยน)
# ============================================
fx_fetch_start = start - datetime.timedelta(days=60)
usdkrw_raw = web.DataReader("DEXKOUS", "fred", fx_fetch_start, end)
usdkrw_raw.columns = ["USDKRW"]
usdkrw_daily_filled = usdkrw_raw.asfreq("D").ffill()
usdkrw_monthly = usdkrw_daily_filled.resample("MS").last()

# ============================================
# 1. GDP รายไตรมาส (FRED โดยตรง — ยืนยันแล้วว่ายังอัปเดตปกติ ไม่ต้องเปลี่ยน)
# ============================================
gdp_quarterly = web.DataReader("NGDPRSAXDCKRQ", "fred", start, end)
gdp_quarterly.columns = ["GDP_Korea"]
gdp_monthly_interp = gdp_quarterly.resample("MS").interpolate(method="linear")

# ============================================
# 2. CPI จาก OECD โดยตรง (แทน FRED KORCPIALLMINMEI ทั้งหมด)
# ครอบคลุมต่อเนื่อง 1996-ปัจจุบัน ไม่มี gap เหมือนของเดิม
# ============================================
cpi_url = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.SDD.TPS,DSD_PRICES@DF_PRICES_ALL,1.0/KOR.M.N.CPI.._T.N.GY+_Z"
    "?startPeriod=1996-01&dimensionAtObservation=AllDimensions"
)
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
response = requests.get(cpi_url, headers=headers)
response.raise_for_status()
cpi_raw = pd.read_csv(io.StringIO(response.text))

cpi_index_only = cpi_raw[cpi_raw["TRANSFORMATION"] == "_Z"].copy()
cpi_korea = cpi_index_only[["TIME_PERIOD", "OBS_VALUE"]].copy()
cpi_korea.columns = ["TIME_PERIOD", "CPI_Korea"]
cpi_korea["TIME_PERIOD"] = pd.to_datetime(cpi_korea["TIME_PERIOD"])
cpi_korea = cpi_korea.set_index("TIME_PERIOD").sort_index()

# ============================================
# 3. GDP_Proxy_Monthly จาก OECD (Reference series (GDP), Normalized)
# ============================================
gdp_proxy_url = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.SDD.STES,DSD_STES@DF_CLI,4.1/KOR.M.RS...NOR..."
    "?startPeriod=1996-01&endPeriod=2026-09&dimensionAtObservation=AllDimensions"
)
response2 = requests.get(gdp_proxy_url, headers=headers)
response2.raise_for_status()
gdp_proxy_raw = pd.read_csv(io.StringIO(response2.text))

gdp_proxy_monthly = gdp_proxy_raw[["TIME_PERIOD", "OBS_VALUE"]].copy()
gdp_proxy_monthly.columns = ["TIME_PERIOD", "GDP_Proxy_Monthly"]
gdp_proxy_monthly["TIME_PERIOD"] = pd.to_datetime(gdp_proxy_monthly["TIME_PERIOD"])
gdp_proxy_monthly = gdp_proxy_monthly.set_index("TIME_PERIOD").sort_index()

# ============================================
# รวมทุกตาราง
# ============================================
df = (
    gdp_monthly_interp
    .join(cpi_korea, how="outer")
    .join(gdp_proxy_monthly, how="outer")
    .join(usdkrw_monthly, how="outer")
)
df = df.loc["1996-01-01":]
df["GDP_Korea_USD_Million"] = df["GDP_Korea"] / df["USDKRW"]

new_order = ["GDP_Korea", "USDKRW", "GDP_Korea_USD_Million", "GDP_Proxy_Monthly", "CPI_Korea"]
df = df[new_order]

print("=== วันที่ข้อมูลแรกสุด/ล่าสุดของแต่ละคอลัมน์ ===")
for col in df.columns:
    print(f"{col:25s}: เริ่ม {df[col].first_valid_index()}   ล่าสุด {df[col].last_valid_index()}")

display(df)

ModuleNotFoundError: No module named 'pandas'

weekly_full_final.py

In [ ]:
!pip install finance-datareader yfinance -q

import pandas as pd
import numpy as np
import pandas_datareader.data as web
import FinanceDataReader as fdr
import yfinance as yf
import requests
import io
import datetime
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

start = datetime.datetime(1996, 1, 1)
end = datetime.datetime.today()
WEEK_ANCHOR = "W-FRI"

oecd_headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}

# ============================================
# 1. Unemployment Rate จาก OECD (แทน FRED เดิม — ไม่มี gap)
# ============================================
unemployment_url = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.SDD.STES,DSD_KEI@DF_KEI,4.0/KOR.M.UNEMP...Y."
    "?startPeriod=1996-01&endPeriod=2026-09&dimensionAtObservation=AllDimensions"
)
response = requests.get(unemployment_url, headers=oecd_headers)
response.raise_for_status()
unemployment_raw = pd.read_csv(io.StringIO(response.text))

unemployment_korea = unemployment_raw[["TIME_PERIOD", "OBS_VALUE"]].copy()
unemployment_korea.columns = ["TIME_PERIOD", "Unemployment_Korea"]
unemployment_korea["TIME_PERIOD"] = pd.to_datetime(unemployment_korea["TIME_PERIOD"])
unemployment_korea = unemployment_korea.set_index("TIME_PERIOD").sort_index()

# ⚠️ ห้าม resample("W-FRI").interpolate() ตรงๆ จากข้อมูลรายเดือน
# เพราะ index รายเดือน (วันที่ 1) ไม่เคยตรงกับ index รายสัปดาห์ (วันศุกร์) เลย
# ทำให้ pandas หาจุดอ้างอิงไป interpolate ไม่ได้ กลายเป็นค่าลากซ้ำแทน (เหมือนปัญหา USDKRW ก่อนหน้า)
#
# วิธีแก้แบบเดียวกับ USDKRW: เติมเป็นปฏิทินรายวันเต็มก่อน (ใช้ time-based interpolation
# เพราะแต่ละเดือนมีจำนวนวันไม่เท่ากัน 28-31 วัน) แล้วค่อยลดเป็นรายสัปดาห์
unemployment_daily = unemployment_korea.asfreq("D").interpolate(method="time")
unemployment_weekly = unemployment_daily.resample(WEEK_ANCHOR).last()

# ============================================
# 2. Policy Interest Rate (FRED)
# ⚠️ ข้อมูลรายเดือน 1 จุด (เช่น 2026-06-01) หมายถึง "อัตราตลอดทั้งเดือนนั้น"
# ต้องขยายปฏิทินให้ครอบคลุมถึง "สิ้นเดือน" ของจุดข้อมูลสุดท้าย ไม่ใช่แค่ asfreq("D")
# เฉยๆ (ซึ่งจะหยุดแค่วันที่ 1 ของเดือนสุดท้าย ทำให้ขาดอีก ~29 วันที่ควรมีค่าจริง)
# ============================================
rate_korea = web.DataReader("INTDSRKRM193N", "fred", start, end)
rate_korea.columns = ["PolicyRate_Korea"]

last_known_month_end = rate_korea.index.max() + pd.offsets.MonthEnd(0)
daily_index = pd.date_range(start=rate_korea.index.min(), end=last_known_month_end, freq="D")
rate_daily = rate_korea.reindex(daily_index).ffill()
rate_weekly = rate_daily.resample(WEEK_ANCHOR).last()

# ============================================
# 3. KOSPI (fdr — ครอบคลุมเต็ม ไม่ต้องเปลี่ยน) + Samsung/SK_Hynix (yfinance — ย้อนได้ถึง 2000)
# ============================================
kospi = fdr.DataReader("KS11", start, end)[["Close"]].rename(columns={"Close": "KOSPI"})
kospi_weekly = kospi.resample(WEEK_ANCHOR).last()

def fetch_yf_close(ticker: str) -> pd.Series:
    # auto_adjust=False: ใช้ราคาปิดดิบ ไม่ปรับย้อนหลังตาม split/dividend/capital reduction
    # (auto_adjust=True เคยทำให้ SK_Hynix ได้ค่าติดลบ เพราะบริษัทมีประวัติลดทุน/ปรับโครงสร้างหนี้ซับซ้อนช่วงต้น 2000s)
    raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if raw.empty:
        return pd.Series(dtype=float)
    close = raw["Close"].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw["Close"]
    weekly = close.resample(WEEK_ANCHOR).last()

    # Safety check: ราคาหุ้นห้ามติดลบเด็ดขาด ถ้าเจอให้แปลงเป็น NaN แทนการปล่อยผ่าน
    n_negative = (weekly < 0).sum()
    if n_negative > 0:
        print(f"[Warning] {ticker}: พบค่าติดลบ {n_negative} จุด (ข้อมูลผิดปกติจากแหล่งข้อมูล) -> แปลงเป็น NaN")
        weekly = weekly.where(weekly >= 0)

    return weekly

samsung_weekly = fetch_yf_close("005930.KS").to_frame(name="Samsung")
sk_hynix_weekly = fetch_yf_close("000660.KS").to_frame(name="SK_Hynix")

# ============================================
# 4. SOX Index (yfinance)
# ============================================
sox_weekly = fetch_yf_close("^SOX").to_frame(name="SOX_Index")

# ============================================
# 5. Nasdaq 100 Index (yfinance)
# ============================================
ndx_weekly = fetch_yf_close("^NDX").to_frame(name="Nasdaq100_Index")

# ============================================
# 6. US 10-Year Treasury Yield (FRED)
# ============================================
us10y = web.DataReader("DGS10", "fred", start, end)
us10y.columns = ["US10Y_Yield"]
us10y_weekly = us10y.resample(WEEK_ANCHOR).last().ffill()

# ============================================
# 7. DXY Index (yfinance)
# ============================================
dxy_weekly = fetch_yf_close("DX-Y.NYB").to_frame(name="DXY_Index")

# ============================================
# 7b. Brent Crude Oil — proxy ของ geopolitical shock (Macro 1)
# ⚠️ เปลี่ยนจาก yfinance (BZ=F) มาใช้ FRED (DCOILBRENTEU) แทน
# เพราะ BZ=F เป็นสัญญา futures เฉพาะแบบที่เพิ่งเริ่มเทรดปี 2007
# ส่วน FRED เก็บราคาน้ำมัน Brent ย้อนได้ถึงปี 1987 (ยาวกว่ามาก)
# ============================================
oil_fetch_start = start - datetime.timedelta(days=60)  # buffer กัน edge case แบบเดียวกับ USDKRW
oil_raw = web.DataReader("DCOILBRENTEU", "fred", oil_fetch_start, end)
oil_raw.columns = ["Brent_Oil"]
oil_daily_filled = oil_raw.asfreq("D").ffill()
oil_weekly = oil_daily_filled.resample(WEEK_ANCHOR).last()

# ============================================
# 7c. VIX Index (yfinance) — วัดความตื่นตระหนกของตลาดโลก (Synthesis / Micro 2 proxy)
# ============================================
vix_weekly = fetch_yf_close("^VIX").to_frame(name="VIX_Index")

# ============================================
# 7d. Gold Price — proxy เงินทุนหนีความเสี่ยง (Macro 2)
# ⚠️ FRED (GOLDAMGBD228NLBM/GOLDPMGBD228NLBM) ถูกยกเลิกไปแล้ว (ยืนยันแล้ว)
# yfinance (GC=F) ก็สั้นเกินไป (เริ่มราวปี 2000)
# ใช้ Stooq แทน — ผ่าน pandas_datareader ตัวเดิม ไม่ต้องติดตั้งใหม่ ย้อนได้ยาวมาก
# ============================================
# ============================================
# 7d. Gold Price — proxy เงินทุนหนีความเสี่ยง (Macro 2)
# ⚠️ ลองหาแหล่งที่ย้อนได้ยาวกว่านี้มาหมดแล้ว:
#   - FRED (GOLDAMGBD228NLBM/GOLDPMGBD228NLBM) -> ถูกยกเลิกถาวร
#   - Stooq (XAUUSD) -> เปลี่ยนนโยบายมี.ค. 2026 ต้องมี API key (ขอผ่าน captcha ด้วยมือ)
# เหลือ yfinance (GC=F) เป็นทางเลือกสุดท้ายที่ใช้งานได้จริง
# ข้อจำกัดที่ยอมรับ: ข้อมูลเริ่มได้แค่ราวปี 2000 (ไม่ใช่ 1996 เต็ม)
# ============================================
gold_weekly = fetch_yf_close("GC=F").to_frame(name="Gold_Price")

# ============================================
# 8. Market Concentration (CR5) — ราคาดึงจาก yfinance ให้สอดคล้องกับ Samsung/SK_Hynix ด้านบน
# ============================================
top5_tickers = pd.Series({
    "Samsung": "005930",
    "SK_Hynix": "000660",
    "LG_Energy": "373220",          # IPO 2022 -> ก่อนหน้านั้นจะเป็น NaN
    "Samsung_Biologics": "207940",  # IPO 2016 -> ก่อนหน้านั้นจะเป็น NaN
    "Hyundai_Motor": "005380",
})

SHARES_OUTSTANDING_FALLBACK = pd.Series({
    "Samsung": 5_969_782_550,
    "SK_Hynix": 728_002_365,
    "LG_Energy": 234_000_000,
    "Samsung_Biologics": 71_174_000,
    "Hyundai_Motor": 140_624_000,
})
TOTAL_MARKET_CAP_FALLBACK_MILLION_KRW = 2_500_000_000

def fetch_weekly_close(code: str) -> pd.Series:
    """ดึงราคาปิดรายวันจาก yfinance (.KS suffix) แล้ว resample เป็นรายสัปดาห์"""
    try:
        return fetch_yf_close(f"{code}.KS")
    except Exception:
        return pd.Series(dtype=float)

shares_outstanding = SHARES_OUTSTANDING_FALLBACK.copy()
total_market_cap_snapshot = TOTAL_MARKET_CAP_FALLBACK_MILLION_KRW
used_live_snapshot = False

try:
    marcap_listing = fdr.StockListing("KRX-MARCAP")
    marcap_listing["Code"] = marcap_listing["Code"].astype(str).str.zfill(6)
    marcap_listing = marcap_listing.set_index("Code")

    snapshot = marcap_listing.reindex(top5_tickers.values)[["Marcap", "Close"]].copy()
    snapshot.index = top5_tickers.index
    live_shares = snapshot["Marcap"] / snapshot["Close"]

    if live_shares.notna().all():
        shares_outstanding = live_shares
        total_market_cap_snapshot = marcap_listing["Marcap"].sum()
        used_live_snapshot = True
        print("[Info] ดึง live market cap snapshot จาก KRX สำเร็จ ใช้ตัวเลขนี้แทนค่า fallback")
    else:
        print("[Warning] live snapshot ดึงมาได้แต่ไม่ครบ 5 บริษัท -> ใช้ค่า fallback (hardcode) แทน")
except Exception as e:
    print(f"[Warning] ดึง live market cap snapshot ไม่สำเร็จ ({e}) -> ใช้ค่า fallback (hardcode) แทน")

price_weekly_df = top5_tickers.apply(fetch_weekly_close).T
price_weekly_df.columns = top5_tickers.index

marcap_weekly_df = price_weekly_df.multiply(shares_outstanding, axis=1)
top5_marcap_sum = marcap_weekly_df.sum(axis=1, skipna=True)
cr5_company_count = marcap_weekly_df.notna().sum(axis=1)

cr5_weekly = top5_marcap_sum.to_frame(name="CR5_Concentration")
cr5_weekly["CR5_Concentration"] = cr5_weekly["CR5_Concentration"] / total_market_cap_snapshot
cr5_weekly["CR5_Company_Count"] = cr5_company_count

print(f"[Info] ใช้ shares_outstanding จาก {'live snapshot' if used_live_snapshot else 'ค่า fallback (hardcode)'}")

# ============================================
# รวมทุกตารางเข้าด้วยกัน
# ============================================
df = (
    unemployment_weekly
    .join(rate_weekly, how="outer")
    .join(kospi_weekly, how="outer")
    .join(samsung_weekly, how="outer")
    .join(sk_hynix_weekly, how="outer")
    .join(sox_weekly, how="outer")
    .join(ndx_weekly, how="outer")
    .join(us10y_weekly, how="outer")
    .join(dxy_weekly, how="outer")
    .join(oil_weekly, how="outer")
    .join(vix_weekly, how="outer")
    .join(gold_weekly, how="outer")
    .join(cr5_weekly, how="outer")
)

df = df.loc["1996-01-01":]

new_order = [
    "Unemployment_Korea", "PolicyRate_Korea",
    "KOSPI", "Samsung", "SK_Hynix", "CR5_Concentration",
    "SOX_Index", "Nasdaq100_Index", "US10Y_Yield", "DXY_Index",
    "Brent_Oil", "VIX_Index", "Gold_Price",
    "CR5_Company_Count",
]
df = df[new_order]

print("\n=== วันที่ข้อมูลแรกสุด/ล่าสุดของแต่ละคอลัมน์ ===")
for col in df.columns:
    print(f"{col:22s}: เริ่ม {df[col].first_valid_index()}   ล่าสุด {df[col].last_valid_index()}")

print()
print("จำนวนแถวทั้งหมด (สัปดาห์):", len(df))

display(df)

[Info] ดึง live market cap snapshot จาก KRX สำเร็จ ใช้ตัวเลขนี้แทนค่า fallback
[Info] ใช้ shares_outstanding จาก live snapshot

=== วันที่ข้อมูลแรกสุด/ล่าสุดของแต่ละคอลัมน์ ===
Unemployment_Korea    : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-07-03 00:00:00
PolicyRate_Korea      : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-07-03 00:00:00
KOSPI                 : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-09-18 00:00:00
Samsung               : เริ่ม 2000-01-07 00:00:00   ล่าสุด 2026-09-18 00:00:00
SK_Hynix              : เริ่ม 2000-01-07 00:00:00   ล่าสุด 2026-09-18 00:00:00
CR5_Concentration     : เริ่ม 2000-01-07 00:00:00   ล่าสุด 2026-09-18 00:00:00
SOX_Index             : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-09-18 00:00:00
Nasdaq100_Index       : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-09-18 00:00:00
US10Y_Yield           : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-09-11 00:00:00
DXY_Index             : เริ่ม 1996-01-05 00:00:00   ล่าสุด 2026-09-18 00:00:00
Brent_Oil             : เริ่ม 199

,Unemployment_Korea,PolicyRate_Korea,KOSPI,Samsung,SK_Hynix,CR5_Concentration,SOX_Index,Nasdaq100_Index,US10Y_Yield,DXY_Index,Brent_Oil,VIX_Index,Gold_Price,CR5_Company_Count
1996-01-05,1.900000,5.00,856.62,NaN,NaN,NaN,196.160004,565.140015,5.69,85.059998,19.50,13.580000,NaN,NaN
1996-01-12,1.900000,5.00,878.64,NaN,NaN,NaN,179.770004,552.710022,5.75,85.059998,17.58,14.230000,NaN,NaN
1996-01-19,1.900000,5.00,848.07,NaN,NaN,NaN,173.869995,564.640015,5.54,86.750000,17.53,12.700000,NaN,NaN
1996-01-26,1.900000,5.00,867.38,NaN,NaN,NaN,187.889999,577.049988,5.65,87.639999,16.70,12.000000,NaN,NaN
1996-02-02,1.903448,5.00,881.48,NaN,NaN,NaN,195.970001,601.419983,5.66,87.010002,17.05,13.230000,NaN,NaN
1996-02-09,1.927586,5.00,881.88,NaN,NaN,NaN,199.630005,623.010010,5.66,86.870003,17.23,14.630000,NaN,NaN
1996-02-16,1.951724,5.00,878.03,NaN,NaN,NaN,193.410004,619.039978,5.76,86.120003,18.20,15.370000,NaN,NaN
1996-02-23,1.975862,5.00,876.55,NaN,NaN,NaN,205.029999,642.590027,5.97,85.449997,18.85,14.780000,NaN,NaN
1996-03-01,2.000000,5.00,852.83,NaN,NaN,NaN,178.759995,604.760010,5.99,86.529999,18.55,16.719999,NaN,NaN
1996-03-08,2.000000,5.00,849.15,NaN,NaN,NaN,172.710007,591.710022,6.41,86.900002,18.85,20.700001,NaN,NaN
